# Safety & Governance Module — NYC 311 Data Exploration

**Author:** Arbaz Khalid — Safety & Governance Module  
**Project:** Plexa — Multi-Modal Smart City Digital Twin and Cascading Resilience Engine  
**Status:** Exploration — checking what NYC 311 data can give the Safety and Governance parts

## Overview

This notebook is part of the Safety & Citizen Governance module of Plexa. While the Climate module relates aggregate rainfall to citywide complaint volumes, this module works on individual citizen reports: Safety decides whether a report is a real hazard, and Governance extracts what and where.

## Purpose (exploration only)

1. Query the NYC 311 Socrata Open Data API for the Hurricane Ida window (Aug 28 – Sep 3, 2021).
2. List the available fields and complaint types.
3. Check whether any field holds real citizen free text.
4. Check how many reports carry coordinates.
5. Make a first draft of the hazard labels (flood, power_outage, road_blockage, non_hazard) and look at the class balance.

Cleaning, final labelling and the incident node export are done in later notebooks.

In [1]:
import os
import requests
import pandas as pd

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"   # NYC 311 Service Requests
START = "2021-08-28T00:00:00"
END   = "2021-09-03T00:00:00"
WHERE = f"created_date between '{START}' and '{END}'"

def soda_get(params):
    r = requests.get(URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

## Step 1 — Fields

Pulled 20 sample reports to list all available columns and check for a free-text field.

In [2]:
sample = pd.DataFrame(soda_get({"$where": WHERE, "$limit": 20}))
print(f"{sample.shape[1]} columns")
print(list(sample.columns))
sample.head(3).T

36 columns
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location', 'descriptor_2', 'facility_type']


,0,1,2
unique_key,51734887,51742446,51731090
created_date,2021-09-03T00:00:00.000,2021-09-03T00:00:00.000,2021-09-03T00:00:00.000
closed_date,2021-09-09T00:00:01.000,2021-09-09T00:00:01.000,2021-09-07T00:00:01.000
agency,DOHMH,DOHMH,DOHMH
agency_name,Department of Health and Mental Hygiene,Department of Health and Mental Hygiene,Department of Health and Mental Hygiene
complaint_type,Food Poisoning,Food Poisoning,Food Poisoning
descriptor,1 or 2,1 or 2,1 or 2
location_type,Restaurant/Bar/Deli/Bakery,Restaurant/Bar/Deli/Bakery,Restaurant/Bar/Deli/Bakery
incident_zip,10001,11418,11238
incident_address,224 WEST 35 STREET,104-02 JAMAICA AVENUE,771 WASHINGTON AVENUE


## Step 2 — Complaint types in the Ida window

Count every complaint type in the window, then search for hazard-related words without caring about upper or lower case.

In [4]:
types = pd.DataFrame(soda_get({
    "$where": WHERE,
    "$select": "complaint_type, count(*) as total",
    "$group": "complaint_type",
    "$order": "total DESC",
    "$limit": 500,
}))
types["total"] = types["total"].astype(int)
print(f"{len(types)} complaint types, {types['total'].sum()} reports in window")
types.head(40)

153 complaint types, 63709 reports in window


,complaint_type,total
0,Request Large Bulky Item Collection,6411
1,Noise - Residential,5621
2,Sewer,5208
3,Illegal Parking,4939
4,Noise - Street/Sidewalk,4072
5,Blocked Driveway,2506
6,UNSANITARY CONDITION,2259
7,WATER LEAK,2164
8,Noise - Vehicle,1670
9,PAINT/PLASTER,1498


In [5]:
pattern = "flood|sewer|basin|drain|water|electric|light|signal|tree|street|road|pothole|hydrant|power"
types[types["complaint_type"].str.contains(pattern, case=False, na=False)]

,complaint_type,total
2,Sewer,5208
4,Noise - Street/Sidewalk,4072
7,WATER LEAK,2164
10,Damaged Tree,1293
11,Water System,1287
13,Traffic Signal Condition,1113
15,Street Condition,1050
19,Street Light Condition,801
21,HEAT/HOT WATER,793
28,ELECTRIC,623


## Step 3 — Descriptors of candidate hazard types

Complaint types are broad. The `descriptor` column gives the detail, so we count descriptors for the candidate types to see which ones are real hazards.

In [6]:
candidates = ["Sewer", "Water System", "Standing Water", "Street Condition", "Damaged Tree",
              "Traffic Signal Condition", "Street Light Condition", "ELECTRIC", "Electrical"]
quoted = ", ".join(f"'{c}'" for c in candidates)

desc = pd.DataFrame(soda_get({
    "$where": f"{WHERE} AND complaint_type in ({quoted})",
    "$select": "complaint_type, descriptor, count(*) as total",
    "$group": "complaint_type, descriptor",
    "$order": "complaint_type, total DESC",
    "$limit": 500,
}))
desc["total"] = desc["total"].astype(int)
print(f"{len(desc)} complaint_type + descriptor combinations")
desc

128 complaint_type + descriptor combinations


,complaint_type,descriptor,total
0,Damaged Tree,Branch or Limb Has Fallen Down,497
1,Damaged Tree,Entire Tree Has Fallen Down,378
2,Damaged Tree,Branch Cracked and Will Fall,187
3,Damaged Tree,Tree Leaning/Uprooted,133
4,Damaged Tree,Tree Alive - in Poor Condition,77
...,...,...,...
123,Water System,Hyd Valve Box Cover Missing (WV2),19
124,Water System,Hydrant Locking Device Request (Use Comments) ...,16
125,Water System,LOW WATER PRESSURE - WLWP,15
126,Water System,Other Water Problem (Use Comments) (WZZ),11


In [7]:
for ctype, group in desc.groupby("complaint_type"):
    print(f"\n=== {ctype}  (total {group['total'].sum()})")
    print(group.sort_values("total", ascending=False).head(8)[["descriptor", "total"]].to_string(index=False))


=== Damaged Tree  (total 1293)
                    descriptor  total
Branch or Limb Has Fallen Down    497
   Entire Tree Has Fallen Down    378
  Branch Cracked and Will Fall    187
         Tree Leaning/Uprooted    133
Tree Alive - in Poor Condition     77
              Tree Trunk Split     21

=== ELECTRIC  (total 623)
   descriptor  total
     LIGHTING    153
OUTLET/SWITCH    153
 POWER OUTAGE    138
  NO LIGHTING    117
       WIRING     62

=== Electrical  (total 36)
                         descriptor  total
Electrical Wiring Defective/Exposed     36

=== Sewer  (total 5208)
                                      descriptor  total
                Sewer Backup (Use Comments) (SA)   3548
                            Street Flooding (SJ)    723
Catch Basin Clogged/Flooding (Use Comments) (SC)    603
         Manhole Cover Missing (Emergency) (SA3)     76
           Manhole Overflow (Use Comments) (SA1)     67
        Other Sewer Problem (Use Comments) (SZZ)     60
             Culve

## Step 4 — Draft weak labels from complaint type + descriptor

First-draft rules, to be reviewed. They label a report by its complaint type and descriptor, which are category fields and not citizen text. Used here only to look at class balance.

In [8]:
RULES = [
    ("Sewer", "Sewer Backup", "flood"),
    ("Sewer", "Street Flooding", "flood"),
    ("Sewer", "Catch Basin", "flood"),
    ("Sewer", "Manhole Overflow", "flood"),
    ("Water System", "Excessive Water In Basement", "flood"),
    ("ELECTRIC", "POWER OUTAGE", "power_outage"),
    ("ELECTRIC", "NO LIGHTING", "power_outage"),
    ("Damaged Tree", "Entire Tree Has Fallen", "road_blockage"),
    ("Damaged Tree", "Branch or Limb Has Fallen", "road_blockage"),
    ("Damaged Tree", "Tree Leaning/Uprooted", "road_blockage"),
    ("Street Condition", "Cave-in", "road_blockage"),
]

def weak_label(ctype, descriptor):
    for rule_type, keyword, label in RULES:
        if ctype == rule_type and keyword.lower() in str(descriptor).lower():
            return label
    return "non_hazard"

desc["weak_label"] = desc.apply(lambda r: weak_label(r["complaint_type"], r["descriptor"]), axis=1)

TOTAL = types["total"].sum()
hazard = desc[desc["weak_label"] != "non_hazard"].groupby("weak_label")["total"].sum()
summary = hazard.to_frame("count")
summary.loc["non_hazard"] = TOTAL - hazard.sum()
summary["percent"] = (100 * summary["count"] / TOTAL).round(1)
summary

,count,percent
weak_label,,
flood,5142,8.1
power_outage,255,0.4
road_blockage,1216,1.9
non_hazard,57096,89.6


## Step 5 — Location coverage

Check how many reports carry coordinates, an address, cross streets and a borough. If most do, 311 needs little location extraction and the Governance extraction work is mainly for text sources such as social posts.

In [9]:
cols = ["latitude", "longitude", "incident_address", "street_name", "cross_street_1", "borough"]
select = "count(*) as total, " + ", ".join(f"count({c}) as has_{c}" for c in cols)

def coverage(extra_filter=""):
    row = soda_get({"$where": WHERE + extra_filter, "$select": select})[0]
    total = int(row["total"])
    return total, {c: round(100 * int(row[f"has_{c}"]) / total, 1) for c in cols}

for name, extra in [("All reports", ""), ("Sewer reports only", " AND complaint_type = 'Sewer'")]:
    total, pct = coverage(extra)
    print(f"\n{name}: {total} reports")
    for c, p in pct.items():
        print(f"  {c:18s} filled: {p}%")


All reports: 63709 reports
  latitude           filled: 97.7%
  longitude          filled: 97.7%
  incident_address   filled: 95.0%
  street_name        filled: 95.0%
  cross_street_1     filled: 74.5%
  borough            filled: 100.0%

Sewer reports only: 5208 reports
  latitude           filled: 99.4%
  longitude          filled: 99.4%
  incident_address   filled: 96.8%
  street_name        filled: 96.8%
  cross_street_1     filled: 96.7%
  borough            filled: 100.0%


## Findings (Ida window, Aug 28 – Sep 3, 2021)

- 63,709 reports across 153 complaint types; 36 columns.
- No citizen free-text column. `resolution_description` is the agency's reply, not the citizen's words.
- Flood reports are hidden in descriptors (Sewer Backup, Street Flooding, Catch Basin, Excessive Water In Basement). No complaint type is named "flood".
- `Standing Water` is mostly pools/containers (not flooding); `Water System` is mostly hydrants and leaks.
- Draft weak labels: flood 8.1%, power_outage 0.4%, road_blockage 1.9%, non_hazard 89.6% — strong class imbalance.
- power_outage has only 255 reports; may need extra data or merging.
- Labels come from category fields, not language, so a classifier trained on them would only learn the category rule.
- Coordinates: 97.7% of all reports and 99.4% of Sewer reports have latitude/longitude; 95–97% have a street address. 311 needs no text-based location extraction. Location extraction is needed only for text sources (e.g. tweets) that carry no coordinates.
